In [37]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import mediapipe as mp

### Các bộ phân được tách
head 0

shoulder 11, 12

elbow 13, 14

wrist 15, 16

hip 23, 24

knee 25, 26

ankle 27, 28

In [38]:
video_path = "gait.mp4"
model_path = "pose_landmarker_heavy.task"

In [39]:
KEYPOINTS = [0, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]

BODY_PARTS = {
    'head':     [0],
    'shoulder': [11, 12],
    'elbow':    [13, 14],
    'wrist':    [15, 16],
    'hip':      [23, 24],
    'knee':     [25, 26],
    'ankle':    [27, 28],
}

GROUP = {
    0: 'head',
    11: 'left shoulder', 12: 'right shoulder',
    13: 'left elbow', 14: 'right elbow',
    15: 'left wrist', 16: 'right wrist',
    23: 'left hip', 24: 'right hip',
    25: 'left knee', 26: 'right knee',
    27: 'left ankle', 28: 'right ankle'
}

### Pose landmarks
Tọa độ (x, y) chuẩn hóa theo kích thước ảnh/video (từ 0 đến 1)
z ở đây là độ sâu tương đối so với hông, nhưng scale theo chiều rộng ảnh

Phụ thuộc vào vị trí và khoảng cách của người trong khung hình
### Pose world landmarks
Tọa độ 3D thực tế (x, y, z) tính bằng mét
Gốc tọa độ (0, 0, 0) nằm ở trung điểm của hông

In [40]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

base_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1)


cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

def draw_landmarks(frame, result):
    if not result.pose_landmarks:
        return frame
    
    frame_copy = frame.copy()
    (h, w) = frame_copy.shape[:2]

    point_color = (0, 0, 255)


    for idx in KEYPOINTS:
        landmark = result.pose_landmarks[0][idx]

        if landmark.presence < 0.7 or landmark.visibility < 0.7:
            continue
        
        px = int(landmark.x * w)
        py = int(landmark.y * h)

        cv2.circle(frame_copy, (px, py), 6, point_color, -1)

    return frame_copy

extracted_data = []
def extract_data(result, frame_idx):
    if not result.pose_world_landmarks:
        return
    
    data = result.pose_world_landmarks[0]

    for landmark_id, landmark in enumerate(data):
        if landmark_id not in KEYPOINTS:
            continue

        extracted_data.append({
            'frame': frame_idx,
            'part': GROUP[landmark_id],
            'x': landmark.x,
            'y': landmark.y,
            'z': landmark.z,
            'visibility': landmark.visibility,
            'presence': landmark.presence
        })



with PoseLandmarker.create_from_options(base_options) as landmarker:
    frame_idx = 0
    rows = []

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        timestamp_ms = int((frame_idx / fps) * 1000)
        image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        result = landmarker.detect_for_video(image, timestamp_ms)

        if result.pose_landmarks:
            for landmark_id, landmark in enumerate(result.pose_landmarks[0]):
                if landmark.presence > 0.5 and landmark.visibility > 0.5:
                    print(f"Frame {frame_idx} | Landmark #{landmark_id}\nx={landmark.x:.3f}\ny={landmark.y:.3f}\nz={landmark.z:.3f}\nvisibility={landmark.visibility}\npresence={landmark.presence}")
        
        drawed_frame = draw_landmarks(frame, result)

        cv2.putText(drawed_frame, f"Frame {frame_idx}", (10, 50), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 0, 255))
        cv2.imshow("Camera", drawed_frame)
        
        extract_data(result, frame_idx)
        
        frame_idx += 1

cap.release()
cv2.destroyAllWindows()

Frame 0 | Landmark #0
x=0.644
y=0.187
z=-0.379
visibility=0.9999668598175049
presence=0.9999935626983643
Frame 0 | Landmark #1
x=0.652
y=0.181
z=-0.363
visibility=0.9999220371246338
presence=0.9999858140945435
Frame 0 | Landmark #2
x=0.657
y=0.181
z=-0.364
visibility=0.9999128580093384
presence=0.999987006187439
Frame 0 | Landmark #3
x=0.661
y=0.181
z=-0.364
visibility=0.9999300241470337
presence=0.9999840259552002
Frame 0 | Landmark #4
x=0.639
y=0.181
z=-0.359
visibility=0.9999431371688843
presence=0.9999796152114868
Frame 0 | Landmark #5
x=0.636
y=0.181
z=-0.360
visibility=0.999951958656311
presence=0.9999825954437256
Frame 0 | Landmark #6
x=0.633
y=0.181
z=-0.360
visibility=0.9999595880508423
presence=0.9999791383743286
Frame 0 | Landmark #7
x=0.667
y=0.183
z=-0.245
visibility=0.9999167919158936
presence=0.9999806880950928
Frame 0 | Landmark #8
x=0.629
y=0.182
z=-0.227
visibility=0.9998400211334229
presence=0.9999663829803467
Frame 0 | Landmark #9
x=0.651
y=0.193
z=-0.330
visibility

In [41]:
df = pd.DataFrame(extracted_data)
df


,frame,part,x,y,z,visibility,presence
0,0,head,-0.012963,-0.612472,-0.380434,0.999967,0.999994
1,0,left shoulder,0.178230,-0.472519,-0.148855,0.999990,0.999998
2,0,right shoulder,-0.118589,-0.528862,-0.055351,0.999972,0.999988
3,0,left elbow,0.223960,-0.279128,-0.127843,0.992569,0.999986
4,0,right elbow,-0.133234,-0.226749,-0.063617,0.968694,0.999967
...,...,...,...,...,...,...,...
8315,639,right hip,0.099323,-0.010490,-0.056409,0.999996,1.000000
8316,639,left knee,-0.070198,0.389765,0.204291,0.909959,0.999979
8317,639,right knee,0.118198,0.328193,0.096948,0.960657,0.999986
8318,639,left ankle,-0.039417,0.716839,0.300790,0.949932,0.999910


In [42]:
df.to_csv("data.csv")